# Open-XTTS-Trainer on Colab (run 4: Phases 6 and 7, all three stages on real weights)

Runs the parts of docs/recipe_design.md (§6: phases 0, 1 and 5) that need the XTTS-v2 weights and a GPU:

| Step | Phase | What it proves |
|---|---|---|
| 1–3 | 0 | Clean environment installs from the pinned dependencies; versions and code fingerprint recorded |
| 4 | 0 | Every upstream artifact matches `artifacts.lock.json`; `mel_stats` three-way check |
| 5 | 0 | Test suite passes in the fresh environment |
| 6 | 1 | Reference voices rebuilt from CMU ARCTIC and verified against the lock from run 2 (clips, licenses, notice) |
| 7 | 5 | Model-gated tests: real `dvae.pth` strict load, upstream weights load into our models, vocoder round-trip canary, real-vocab `[UNK]` check |
| 7b | 6 | Real forced aligner: JFK aligned and cut at pauses, ±1 s cut shifts rejected; sample set prepared and validated with zero edits |
| 7c | 7 | Stage 2 through upstream GPTTrainer with the real weights: eval loss before vs after a short run, export, strict reload, synthesis |
| 7d | 7 | Stage 3: latents from the Stage 2 export, short fine-tune, validation against the starting decoder |
| 7e | 7 | Stage 1: short DVAE fine-tune on real speech, normal and frozen codebook; collapse gate and drift recorded |
| 8 | 1 | *(off by default: proven in runs 1–2)* Smoke run: Whisper large-v3, ECAPA2 and UTMOS all load and score |
| 9 | 1 | *(off by default)* Pilot baseline: English, 12 sentences (2 per category, including short 1–3 word phrases) × 5 voices × 2 seeds = 120 utterances |
| 10 | 1 | *(off by default)* Determinism check: the smoke subset rerun must match exactly (run 1 proved full runs are bit-identical) |
| 11 | 1 | Optional, later: full baseline |
| 12 | — | Evidence bundle to bring back into the repo |

**Licenses you accept by running this:** XTTS-v2 weights are under the Coqui Public Model License (non-commercial only).
ECAPA2 is CC BY-NC 4.0. CMU ARCTIC allows any use if its notice is kept, modifications are marked and the authors are
credited; step 6 saves each speaker's `COPYING` and a `MODIFICATIONS.txt` next to the clips.

**Runtime:** GPU (T4 or better). Expected total on a T4: about 40–60 min, mostly installs and downloads (the pilot is
about 8–10 min). Every step fails loudly; if a cell errors, stop and bring the output back. Don't edit around it.

## 1. GPU and working directory
Results go to Google Drive so a disconnected session doesn't lose them.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/open_xtts_run4'
import os; os.makedirs(RESULTS, exist_ok=True)
print('Results ->', RESULTS)

## 2. Get the code
Upload the working-tree zip (`open-xtts-trainer-worktree.zip`), or, once a branch is pushed, use the clone option.

In [ ]:
%cd /content
import os, zipfile
USE_GIT_BRANCH = None  # e.g. 'phase-0-5' once pushed; None = upload the zip

if USE_GIT_BRANCH:
    !rm -rf open-xtts-trainer && git clone -b {USE_GIT_BRANCH} https://github.com/amitbhakt/open-xtts-trainer.git
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    !rm -rf open-xtts-trainer && mkdir open-xtts-trainer
    zipfile.ZipFile(zip_name).extractall('open-xtts-trainer')
%cd /content/open-xtts-trainer
!ls

## 3. Install pinned dependencies; record the environment and code fingerprint (Phase 0)

In [ ]:
!pip install -q -e . pytest
import json, platform, subprocess, sys, torch, torchaudio, transformers
sys.path.insert(0, '.')
from eval.harness import get_code_fingerprint
env = {
    'python': platform.python_version(),
    'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'transformers': transformers.__version__,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'code_fingerprint': get_code_fingerprint(),
    'pip_freeze': subprocess.check_output(['pip', 'freeze'], text=True).splitlines(),
}
os.makedirs('docs/evidence', exist_ok=True)
json.dump(env, open('docs/evidence/environment_colab.json', 'w'), indent=2)
print({k: v for k, v in env.items() if k != 'pip_freeze'})
print([p for p in env['pip_freeze'] if p.startswith('coqui-tts')])

## 4. Fetch and verify every upstream artifact; three-way `mel_stats` check (Phase 0)
Downloads about 2.1 GB into `xtts_base/`. Exits non-zero on any hash, size or `mel_stats` mismatch.

In [ ]:
!python scripts/fetch_base_artifacts.py --dest xtts_base --check_mel_stats --report docs/evidence/artifact_verification.json
report = json.load(open('docs/evidence/artifact_verification.json'))
assert all(a['sha256_ok'] and a['size_ok'] for a in report['artifacts'].values()), 'artifact mismatch'
assert report['mel_stats_check']['complete'] and report['mel_stats_check']['all_equal'], report['mel_stats_check']
print('Phase 0 artifacts verified; mel_stats three-way check passed.')

## 5. Test suite in the fresh environment (Phase 0)
Takes 3–8 minutes on Colab's CPUs. Model-gated tests run here too, now that `xtts_base/` exists; the round-trip canary waits for step 6's voices.

In [ ]:
!python -m pytest -q 2>&1 | tail -8

## 6. Rebuild and verify the reference voices (Phase 1)
Downloads 5 CMU ARCTIC tarballs (about 400 MB) plus each speaker's license. The clips and licenses must match the lock that run 2 created (`eval/dataset/reference_voices.lock.json`).

In [ ]:
!python scripts/fetch_reference_voices.py  # verifies against the committed lock; fails on any mismatch
lock = json.load(open('eval/dataset/reference_voices.lock.json'))
for sid, e in lock['voices'].items():
    print(f"{sid}: {e['duration_s']}s  {e['utterances_used']}  license={e['license_file']}\n    {e['transcript']}")
!head -12 eval/reference_voices/MODIFICATIONS.txt

## 7. Model-gated tests (Phase 5)
The vocoder round-trip canary, the identity canary (the released model through our export path must come back bit-identical), the real `dvae.pth` strict load, upstream weights loading into our factory models, and real-vocab `[UNK]` detection.

In [ ]:
!python -m pytest -v tests/test_invariants.py tests/test_spec_parity.py tests/test_data_contract.py \
    -k "round_trip or upstream_checkpoint_loads or upstream_dvae_loads or real_vocab or identity_canary" 2>&1 | tail -12

## 7b. Phase 6: forced alignment with the real aligner
Downloads MMS_FA (1.26 GB, CC BY-NC 4.0). Then:
- JFK is aligned and cut at pauses; the clips must re-align cleanly, and a cut moved by ±1 s must be rejected;
- the sample set (LJSpeech + JFK) is prepared and must pass validation with zero manual edits.

In [ ]:
import torchaudio; torchaudio.pipelines.MMS_FA.get_model()
!python -m pytest -v tests/test_prepare.py -k mms 2>&1 | tail -6
!rm -rf sample_audio && python -m xtts_trainer.speech_data --source sample --output_dir sample_audio --vocab xtts_base/vocab.json
!python -m xtts_trainer.data.validate sample_audio/dataset.jsonl --vocab xtts_base/vocab.json --realign --report sample_audio/realign_report.json | tail -3
os.makedirs('docs/evidence/phase6', exist_ok=True)
!cp sample_audio/prep_report.json sample_audio/validation_report.json sample_audio/realign_report.json sample_audio/dataset.jsonl docs/evidence/phase6/
for r in json.load(open('sample_audio/prep_report.json'))['recordings']:
    print(r['input'], {k: r.get(k) for k in ('segments', 'kept_fraction', 'alignment_score', 'pause_issues_before_fix')}, 'dropped:', r.get('dropped'))

## 7c. Phase 7: Stage 2 through upstream GPTTrainer, real weights
A short fine-tune on the sample set from 7b (8 clips: 6 train, 2 eval), starting from the released `model.pth`, loaded strictly.
Records the eval loss before and after, exports the best checkpoint, reloads it into stock `Xtts` with `strict=True`, and synthesizes one sentence.
On a T4 this needs about 7 GB of GPU memory and a few minutes.

In [ ]:
assert os.path.exists('sample_audio/dataset.jsonl'), 'step 7b did not produce the sample set; fix that first'
!python -m xtts_trainer.stage2_gpt.train --dataset sample_audio/dataset.jsonl --base_dir xtts_base \
    --output_dir runs/gpt_smoke --epochs 3 --batch_size 2 --grad_accum_steps 1 --eval_fraction 0.25 > runs_gpt_smoke.log 2>&1
!grep -E "Eval loss|avg_loss:|BEST MODEL|Exported" runs_gpt_smoke.log | tail -12
if not os.path.exists('runs/gpt_smoke/export/training_summary.json'):
    !tail -40 runs_gpt_smoke.log
    raise RuntimeError('Stage 2 did not finish; the log tail is above (full log: runs_gpt_smoke.log)')
os.makedirs('docs/evidence/phase7', exist_ok=True)
!cp runs_gpt_smoke.log docs/evidence/phase7/
summary = json.load(open('runs/gpt_smoke/export/training_summary.json'))
print(json.dumps({k: summary[k] for k in ('eval_loss_before', 'eval_loss_best', 'improved', 'best_step', 'train_clips', 'eval_clips')}, indent=1))
import torch, soundfile as sf
from xtts_trainer.stage2_gpt.export import load_for_inference
xtts = load_for_inference('runs/gpt_smoke/export', 'cuda')
lat, spk = xtts.get_conditioning_latents(audio_path=['eval/reference_voices/arctic_slt.wav'])
torch.manual_seed(42)
wav = xtts.inference('The quick brown fox jumps over the lazy dog.', 'en', lat, spk, temperature=0.7)['wav']
os.makedirs('docs/evidence/phase7', exist_ok=True)
sf.write('docs/evidence/phase7/stage2_smoke_sample.wav', wav, 24000)
!cp runs/gpt_smoke/export/training_summary.json docs/evidence/phase7/
print(f'synthesized {len(wav) / 24000:.2f}s')

## 7d. Phase 7: Stage 3 on real latents (chained from Stage 2)
Extracts latents for the sample set with the Stage 2 export (eval-mode GPT), then fine-tunes the export's decoder for 120 steps:
40 steps of discriminator warm-up, then adversarial training.
The starting decoder's validation error is measured first; `beats_start` says whether the best checkpoint beat it.
A smoke run: 8 clips can show the wiring works and the losses move, not that the decoder improves.

In [ ]:
import gc, torch
for name in ('xtts', 'lat', 'spk', 'wav'):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

from xtts_trainer.config import HiFiGANConfig
from xtts_trainer.stage3_hifigan.extract_latents import extract_latents
from xtts_trainer.stage3_hifigan.train import train_hifigan
extract_latents('sample_audio/dataset.jsonl', 'runs/latents', checkpoint_dir='runs/gpt_smoke/export', device='cuda')
cfg3 = HiFiGANConfig(batch_size=4, epochs=60, disc_warmup_steps=40, val_every_steps=40, sample_every_steps=40)
best3 = train_hifigan('sample_audio/dataset.jsonl', 'runs/latents', 'runs/hifigan_smoke', cfg3,
                      checkpoint_dir='runs/gpt_smoke/export', device='cuda')
ck3 = torch.load(best3, map_location='cpu', weights_only=False)
print({k: ck3[k] for k in ('val_mel_l1', 'reference_val_mel_l1', 'beats_start', 'step')})
!cat runs/hifigan_smoke/metrics.jsonl
os.makedirs('docs/evidence/phase7/stage3', exist_ok=True)
!cp runs/hifigan_smoke/metrics.jsonl docs/evidence/phase7/stage3/ && cp -r runs/hifigan_smoke/samples docs/evidence/phase7/stage3/

## 7e. Phase 7: Stage 1 on real speech (normal and frozen codebook)
Fine-tunes the released `dvae.pth` on the sample set, once normally and once with the codebook frozen.
Records validation loss, active codes, perplexity and code drift. If the collapse gate fires, that is a result, not a failure of this cell.

In [ ]:
from xtts_trainer.config import DVAEConfig
from xtts_trainer.stage1_dvae.train import CodebookCollapseError, train_dvae
stage1 = {}
for mode, frozen in (('normal', False), ('frozen_codebook', True)):
    out = f'runs/dvae_smoke_{mode}'
    try:
        best1 = train_dvae('sample_audio/dataset.jsonl', out, DVAEConfig(batch_size=4, epochs=20, val_every_steps=10,
                                                                         frozen_codebook=frozen),
                           base_checkpoint='xtts_base/dvae.pth', device='cuda')
        ck1 = torch.load(best1, map_location='cpu', weights_only=False)
        stage1[mode] = {'outcome': 'completed', 'best': ck1['val'], 'reference': ck1['reference']}
    except CodebookCollapseError as e:
        stage1[mode] = {'outcome': 'codebook_collapse', 'error': str(e)}
    stage1[mode]['metrics'] = [json.loads(l) for l in open(f'{out}/metrics.jsonl')]
    print(mode, json.dumps({k: v for k, v in stage1[mode].items() if k != 'metrics'}, indent=1))
json.dump(stage1, open('docs/evidence/phase7/stage1_smoke.json', 'w'), indent=2)

## 8. Smoke run: every metric model loads and scores
2 English sentences × 5 voices × 1 seed. Downloads Whisper large-v3, ECAPA2 and UTMOS on first use.

In [ ]:
RUN_PHASE1 = False  # Phase 1 baseline and determinism are proven (docs/evidence/colab_run2)
if RUN_PHASE1:
    !python scripts/run_eval_baseline.py --checkpoint_dir xtts_base --languages en --subset_size 2 --seeds 42 \
        --output eval/baselines/smoke.json
    import statistics, time
    smoke = json.load(open('eval/baselines/smoke.json'))
    print(json.dumps(smoke['summary_by_language'], indent=1))
    print('failures:', smoke['failure_counts'], '| code:', smoke['code_fingerprint'])
    synth_s = [r['synthesis_time_s'] for run in smoke['runs'] for seed in run['per_seed_results'].values() for r in seed]
    print(f'synthesis: {statistics.mean(synth_s):.2f}s per utterance (excludes scoring)')
    PILOT_SENTENCES, PILOT_SEEDS, VOICES = 12, 2, len(smoke['runs'])
    n = PILOT_SENTENCES * PILOT_SEEDS * VOICES
    print(f'pilot = {n} utterances; projected ~{n * statistics.mean(synth_s) * 2.4 / 60:.0f} min (run 1 measured ~2.4x synthesis time end to end)')

## 9. Pilot baseline: English, 12 sentences × 5 voices × 2 seeds
2 sentences from each of the 6 categories (harvard, sibilant, numbers, long_complex, prosody_questions, short).

In [ ]:
if RUN_PHASE1:
    t0 = time.time()
    !python scripts/run_eval_baseline.py --checkpoint_dir xtts_base --languages en \
        --subset_size {PILOT_SENTENCES} --seeds 42 123 --output eval/baselines/pilot_en.json
    !cp eval/baselines/pilot_en.json {RESULTS}/
    print(f'pilot: {(time.time() - t0) / 60:.1f} min')
    pilot = json.load(open('eval/baselines/pilot_en.json'))
    print(json.dumps(pilot['summary_by_language'], indent=1))
    for cat, st in pilot['summary_by_category']['en'].items():
        print(f"{cat:18s} WER {st['wer']['mean']:.3f}  SECS(>=2s) {st['secs']['mean']:.3f} (n={st['secs']['n']})  "
              f"SECS all {st['secs_all']['mean']:.3f}  dur {st['duration_ratio']['mean']:.2f}  too_long {st['failure_counts']['too_long']}")


## 10. Determinism check (Phase 1 exit criterion)
Run 1 showed full runs are bit-identical, so only the smoke subset is rerun and must match exactly.

In [ ]:
if RUN_PHASE1:
    !python scripts/run_eval_baseline.py --checkpoint_dir xtts_base --languages en --subset_size 2 --seeds 42 \
        --output eval/baselines/smoke_rerun.json
    !python scripts/compare_baselines.py eval/baselines/smoke.json eval/baselines/smoke_rerun.json \
        --report docs/evidence/determinism_check.json
    a, b = (json.load(open(f'eval/baselines/{n}.json')) for n in ('smoke', 'smoke_rerun'))
    def utts(m): return [(r['text'], r['seed'], r['asr_text'], r['duration_s']) for run in m['runs'] for s in run['per_seed_results'].values() for r in s]
    same = sum(x == y for x, y in zip(utts(a), utts(b)))
    print(f'identical utterances (text and duration): {same}/{len(utts(a))}')

## 11. Optional, later: full baseline (all eval languages)
About 2.5 hours on a T4, single run (determinism is already established). Leave `RUN_FULL = False` for now.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    !python scripts/run_eval_baseline.py --checkpoint_dir xtts_base --output eval/baselines/base_xtts_full.json
    !cp eval/baselines/base_xtts_full.json {RESULTS}/

## 12. Evidence bundle
Download this zip (also saved to Drive) and bring it back.

In [ ]:
!zip -qr run4_evidence.zip docs/evidence eval/baselines eval/dataset/reference_voices.lock.json \
    eval/dataset/eval_set.json eval/reference_voices
!cp run4_evidence.zip {RESULTS}/
from google.colab import files
files.download('run4_evidence.zip')